In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [18]:
mapping = pd.read_csv("../data/station_district_mapping.csv")
mapping.head()

,district_code,district_name,temp_station,humidity_station,precipitation_station,province
0,101,TAPLEJUNG,Taplejung (°C),Taplejung (%)(Avg),Taplejung (mm),Province 1
1,102,SANKHUWASABHA,Num (°C),Num (%)(Avg),Num (mm),Province 1
2,103,SOLUKHUMBU,Salleri (°C),Salleri (%)(Avg),Salleri (mm),Province 1
3,104,OKHALDHUNGA,Okhaldhunga (°C),Okhaldhunga (%)(Avg),Okhaldhunga (mm),Province 1
4,105,KHOTANG,Diktel (°C),Diktel (%)(Avg),Diktel (mm),Province 1


In [19]:
Integrated = pd.read_csv("../data/integrated_dengue_weather.csv")
Integrated.head()

,Year,Week,District,Cases,temp_lag0,temp_lag1,temp_lag2,temp_lag3,temp_lag4,temp_max_lag0,...,humidity_lag0,humidity_lag1,humidity_lag2,humidity_lag3,humidity_lag4,precipitation_lag0,precipitation_lag1,precipitation_lag2,precipitation_lag3,precipitation_lag4
0,2019,1,Achham,0.0,14.164286,6.562500,NaN,NaN,NaN,20.271429,...,62.807143,78.490000,NaN,NaN,NaN,0.2,5.8,NaN,NaN,NaN
1,2019,2,Achham,0.0,14.164286,6.562500,NaN,NaN,NaN,20.271429,...,62.807143,78.490000,NaN,NaN,NaN,0.2,5.8,NaN,NaN,NaN
2,2019,3,Achham,0.0,14.164286,6.562500,NaN,NaN,NaN,20.271429,...,62.807143,78.490000,NaN,NaN,NaN,0.2,5.8,NaN,NaN,NaN
3,2019,4,Achham,0.0,3.750000,14.164286,6.562500,NaN,NaN,9.750000,...,91.725000,62.807143,78.490000,NaN,NaN,0.0,0.2,5.8,NaN,NaN
4,2019,5,Achham,0.0,13.728571,3.750000,14.164286,6.5625,NaN,20.214286,...,68.464286,91.725000,62.807143,78.49,NaN,2.8,0.0,0.2,5.8,NaN


In [20]:
# check different districts in the dataset and name them all. Do not shorten the list with ....
districts = Integrated['District'].unique()
print(districts)

['Achham' 'Arghakhachi' 'Baglung' 'Baitadi' 'Bajhang' 'Bajura' 'Banke'
 'Bara' 'Bardiya' 'Bhaktapur' 'Bhojpur' 'Chitwan' 'Dadeldhura' 'Dailekh'
 'Dang' 'Darchula' 'Dhading' 'Dhankuta' 'Dhanusha' 'Dolakha' 'Dolpa'
 'Doti' 'Gorkha' 'Gulmi' 'Humla' 'Ilam' 'Jajarkot' 'Jhapa' 'Jumla'
 'Kailali' 'Kalikot' 'Kanchanpur' 'Kapilvastu' 'Kaski' 'Kathmandu'
 'Kavrepalanchowk' 'Khotang' 'Lalitpur' 'Lamjung' 'Mahottari' 'Makwanpur'
 'Manang' 'Morang' 'Mugu' 'Mustang' 'Myagdi' 'Nawalparasi' 'Nuwakot'
 'Okhaldhunga' 'Palpa' 'Panchthar' 'Parbat' 'Parsa' 'Pyuthan' 'Ramechhap'
 'Rasuwa' 'Rautahat' 'Rolpa' 'Rukum East' 'Rukum West' 'Rupandehi'
 'Salyan' 'Sankhuwasabha' 'Saptari' 'Sarlahi' 'Sholukhumbu' 'Sindhuli'
 'Sindhupalchowk' 'Siraha' 'Sunsari' 'Surkhet' 'Syangja' 'Tanahu'
 'Taplejung' 'Terhathum' 'Udayapur']


In [21]:
# write code to delete all the columns from Integrated with names ending with ... _lag1, _lag2, _lag3, _lag4
columns_to_drop = [col for col in Integrated.columns if col.endswith(('_lag1', '_lag2', '_lag3', '_lag4'))]
Integrated.drop(columns=columns_to_drop, inplace=True)
Integrated.head()

,Year,Week,District,Cases,temp_lag0,temp_max_lag0,temp_min_lag0,humidity_lag0,precipitation_lag0
0,2019,1,Achham,0.0,14.164286,20.271429,8.057143,62.807143,0.2
1,2019,2,Achham,0.0,14.164286,20.271429,8.057143,62.807143,0.2
2,2019,3,Achham,0.0,14.164286,20.271429,8.057143,62.807143,0.2
3,2019,4,Achham,0.0,3.750000,9.750000,-2.250000,91.725000,0.0
4,2019,5,Achham,0.0,13.728571,20.214286,7.242857,68.464286,2.8


In [28]:
lag_removed= Integrated.copy()
#save the new dataset to a csv file
lag_removed.to_csv("../data/dengue_weather_lag_removed.csv", index=False)

In [23]:
# Remove forward fill duplicate values in specified columns
# Replace ALL forward-filled duplicate values with NaN (null values)
# A forward-filled value is one that equals the previous non-NaN value

columns_to_clean = ['temp_lag0', 'temp_max_lag0', 'temp_min_lag0', 'humidity_lag0', 'precipitation_lag0']

print("Removing forward fill duplicates...")
print(f"Columns to clean: {columns_to_clean}")
print(f"Total rows before: {len(Integrated)}")

# Group by District to handle duplicates within each district separately
for district in Integrated['District'].unique():
    district_mask = Integrated['District'] == district
    district_indices = Integrated[district_mask].index.tolist()
    
    for col in columns_to_clean:
        if col in Integrated.columns:
            # Track the last valid (non-NaN) value for this district and column
            last_valid_value = None
            
            # Process each row in order
            for idx in district_indices:
                current_value = Integrated.loc[idx, col]
                
                # Skip if already NaN (don't process NaN values)
                if pd.isna(current_value):
                    continue
                
                # If we have a last valid value and current equals it, this is a forward-fill duplicate
                if last_valid_value is not None:
                    # Use numpy.isclose for floating point comparison to handle precision issues
                    if np.isclose(current_value, last_valid_value, equal_nan=False):
                        # This is a forward-fill duplicate, replace with NaN
                        Integrated.loc[idx, col] = np.nan
                        # Don't update last_valid_value - keep tracking the original value
                        continue
                
                # This is a new valid value, update our tracker
                last_valid_value = current_value

print("\nForward fill duplicates removed successfully!")
print(f"Total rows after: {len(Integrated)}")
print("\nSample of cleaned data (first 10 rows):")
Integrated.head(10)


Removing forward fill duplicates...
Columns to clean: ['temp_lag0', 'temp_max_lag0', 'temp_min_lag0', 'humidity_lag0', 'precipitation_lag0']
Total rows before: 27664

Forward fill duplicates removed successfully!
Total rows after: 27664

Sample of cleaned data (first 10 rows):


,Year,Week,District,Cases,temp_lag0,temp_max_lag0,temp_min_lag0,humidity_lag0,precipitation_lag0
0,2019,1,Achham,0.0,14.164286,20.271429,8.057143,62.807143,0.2
1,2019,2,Achham,0.0,NaN,NaN,NaN,NaN,NaN
2,2019,3,Achham,0.0,NaN,NaN,NaN,NaN,NaN
3,2019,4,Achham,0.0,3.750000,9.750000,-2.250000,91.725000,0.0
4,2019,5,Achham,0.0,13.728571,20.214286,7.242857,68.464286,2.8
5,2019,6,Achham,0.0,9.650000,16.566667,2.733333,62.983333,1.5
6,2019,7,Achham,0.0,NaN,NaN,NaN,NaN,NaN
7,2019,8,Achham,0.0,3.500000,11.250000,-4.250000,88.075000,0.0
8,2019,9,Achham,0.0,14.042857,20.142857,7.942857,67.257143,13.3
9,2019,10,Achham,0.0,10.216667,17.666667,2.766667,59.233333,1.5


In [24]:
# # Summary: Check how many duplicates were removed
# print("Summary of forward fill duplicate removal:")
# print("=" * 60)

# columns_to_clean = ['temp_lag0', 'temp_max_lag0', 'temp_min_lag0', 'humidity_lag0', 'precipitation_lag0']

# for col in columns_to_clean:
#     if col in Integrated.columns:
#         null_count = Integrated[col].isna().sum()
#         total_count = len(Integrated[col])
#         null_percentage = (null_count / total_count) * 100
#         print(f"{col:25s}: {null_count:6d} NaN values ({null_percentage:5.2f}% of total)")

# print("\n" + "=" * 60)
# print(f"Total rows in dataset: {len(Integrated)}")
# print(f"Total districts processed: {len(Integrated['District'].unique())}")

In [25]:
# # Weighted Moving Average to Fill NaN Values
# # Uses a weighted average of surrounding valid values, giving more weight to closer values

# def weighted_moving_average_fill(dataframe, columns, window_size=5, weight_type='linear'):
#     """
#     Fill NaN values using weighted moving average method.
    
#     Parameters:
#     -----------
#     dataframe : pd.DataFrame
#         Input dataframe
#     columns : list
#         List of column names to fill
#     window_size : int
#         Size of the window for moving average (default: 5)
#     weight_type : str
#         Type of weighting: 'linear' (closer values have more weight) or 'uniform' (equal weight)
    
#     Returns:
#     --------
#     pd.DataFrame
#         Dataframe with NaN values filled
#     """
#     df_filled = dataframe.copy()
    
#     for col in columns:
#         if col not in df_filled.columns:
#             print(f"Warning: Column '{col}' not found in dataframe")
#             continue
        
#         print(f"\nProcessing column: {col}")
#         initial_nan_count = df_filled[col].isna().sum()
        
#         # Group by District to handle each district separately
#         for district in df_filled['District'].unique():
#             district_mask = df_filled['District'] == district
#             district_df = df_filled[district_mask]
            
#             # Get indices for this district
#             district_indices = district_df.index.tolist()
            
#             # Iterate through each index in the district
#             for i, idx in enumerate(district_indices):
#                 # Skip if value is not NaN
#                 if not pd.isna(df_filled.loc[idx, col]):
#                     continue
                
#                 # Get valid values before and after the NaN
#                 before_indices = district_indices[:i]
#                 after_indices = district_indices[i+1:]
                
#                 # Find the closest valid values
#                 valid_before = []
#                 valid_after = []
                
#                 # Look backwards
#                 for j in range(len(before_indices)-1, -1, -1):
#                     prev_idx = before_indices[j]
#                     val = df_filled.loc[prev_idx, col]
#                     if not pd.isna(val):
#                         valid_before.append((j, val, len(before_indices) - 1 - j))
#                         if len(valid_before) >= window_size:
#                             break
                
#                 # Look forwards
#                 for j in range(len(after_indices)):
#                     next_idx = after_indices[j]
#                     val = df_filled.loc[next_idx, col]
#                     if not pd.isna(val):
#                         valid_after.append((j, val, j))
#                         if len(valid_after) >= window_size:
#                             break
                
#                 # If we have at least one valid value, compute weighted average
#                 if valid_before or valid_after:
#                     weighted_sum = 0
#                     weight_total = 0
                    
#                     # Add values from before
#                     for pos, val, distance in valid_before:
#                         if weight_type == 'linear':
#                             weight = 1.0 / (distance + 1)
#                         else:  # uniform
#                             weight = 1.0
#                         weighted_sum += val * weight
#                         weight_total += weight
                    
#                     # Add values from after
#                     for pos, val, distance in valid_after:
#                         if weight_type == 'linear':
#                             weight = 1.0 / (distance + 1)
#                         else:  # uniform
#                             weight = 1.0
#                         weighted_sum += val * weight
#                         weight_total += weight
                    
#                     # Compute and assign the weighted average
#                     df_filled.loc[idx, col] = weighted_sum / weight_total
        
#         final_nan_count = df_filled[col].isna().sum()
#         filled_count = initial_nan_count - final_nan_count
#         print(f"  Filled: {filled_count} NaN values")
#         print(f"  Remaining NaN: {final_nan_count}")
    
#     return df_filled

# # Apply weighted moving average filling to the specified columns
# columns_to_fill = ['temp_lag0', 'temp_max_lag0', 'temp_min_lag0', 'humidity_lag0', 'precipitation_lag0']

# print("=" * 70)
# print("WEIGHTED MOVING AVERAGE FILLING")
# print("=" * 70)

# Integrated = weighted_moving_average_fill(
#     Integrated, 
#     columns=columns_to_fill,
#     window_size=5,
#     weight_type='linear'
# )

# print("\n" + "=" * 70)
# print("SUMMARY AFTER WEIGHTED MOVING AVERAGE FILLING")
# print("=" * 70)

# for col in columns_to_fill:
#     if col in Integrated.columns:
#         null_count = Integrated[col].isna().sum()
#         total_count = len(Integrated[col])
#         null_percentage = (null_count / total_count) * 100
#         print(f"{col:25s}: {null_count:6d} NaN values ({null_percentage:5.2f}% of total)")

# print("\n" + "=" * 70)
# print(f"Total rows in dataset: {len(Integrated)}")
# print(f"Total districts processed: {len(Integrated['District'].unique())}")
# print("=" * 70)

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
# Filled = pd.read_csv("../data/integrated_filled_WMA.csv")
# Filled.head(10)

In [32]:
# Integrated.head(10)

In [33]:
# old_Integrated = pd.read_csv("../data/integrated_dengue_weather.csv")
# old_Integrated.head(10)